In [1]:
import os
import shutil
import json
import torch
import nltk
import numpy as np
from google.colab import drive
from transformers import BartTokenizer, BartForConditionalGeneration
from datasets import Dataset

# Cấp quyền cho các hàm Numpy (Fix lỗi Unpickling)
try:
    torch.serialization.add_safe_globals([
        np.core.multiarray._reconstruct,
        np.ndarray,
        np.dtype,
    ])
except AttributeError:
    pass

if not hasattr(torch, 'original_load_func'):
    torch.original_load_func = torch.load

def safe_load_override(*args, **kwargs):
    if 'weights_only' in kwargs:
        del kwargs['weights_only']
    return torch.original_load_func(*args, weights_only=False, **kwargs)

torch.load = safe_load_override

os.environ['KAGGLE_USERNAME'] = "phankhaclap"
os.environ['KAGGLE_KEY'] = "0ba946628cb1f5acb76ecd357f590e95"

drive.mount('/content/drive')

# Đường dẫn đến mô hình đã lưu
FINAL_SAVE_PATH = "/content/drive/My Drive/BART_Base_Spider_CRP_FFT"

# 1. TẢI DỮ LIỆU & SPIDER-REALISTIC
if not os.path.exists('spider_data'):
    !pip install -q kaggle
    !kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset
    !unzip -q yale-universitys-spider-10-nlp-dataset.zip -d temp_spider

    if os.path.exists("temp_spider/spider"):
        shutil.move("temp_spider/spider", "spider_data")
        shutil.rmtree('temp_spider')
        !wget -q https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py
        !wget -q https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py
        nltk.download('punkt')
        nltk.download('punkt_tab')

if not os.path.exists('spider_data/spider-realistic.json'):
    !wget -q -O spider_data/spider-realistic.json "https://zenodo.org/records/5205322/files/spider-realistic.json?download=1"

# 2. LOAD MÔ HÌNH TỪ ĐĨA LÊN GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang tải mô hình từ {FINAL_SAVE_PATH} lên {device}...")

tokenizer = BartTokenizer.from_pretrained(FINAL_SAVE_PATH)
model = BartForConditionalGeneration.from_pretrained(FINAL_SAVE_PATH).to(device)
model.eval()

# 3. TIỀN XỬ LÝ SCHEMA VÀ DATASET
with open('spider_data/tables.json', 'r') as f:
    table_data = json.load(f)
schema_map = {db['db_id']: db for db in table_data}

def get_crp_schema(db_id):
    if db_id not in schema_map: return ""
    db = schema_map[db_id]
    ddl_statements = []
    for i, table_name in enumerate(db['table_names_original']):
        col_defs = [f"{c[1]} {db['column_types'][j]}" for j, c in enumerate(db['column_names_original']) if c[0] == i]
        pk_idx = db['primary_keys']
        pks = [db['column_names_original'][j][1] for j in pk_idx if db['column_names_original'][j][0] == i]
        if pks: col_defs.append(f"primary key ({', '.join(pks)})")
        for fk in db['foreign_keys']:
            if db['column_names_original'][fk[0]][0] == i:
                from_col = db['column_names_original'][fk[1]][1]
                to_table = db['table_names_original'][db['column_names_original'][fk[1]][0]]
                to_col = db['column_names_original'][fk[1]][1]
                col_defs.append(f"foreign key ({from_col}) references {to_table}({to_col})")
        ddl_statements.append(f"CREATE TABLE {table_name} ({', '.join(col_defs)});")
    return " ".join(ddl_statements)

def load_spider_unified(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return Dataset.from_dict({
        "question": [item["question"] for item in data],
        "query": [item["query"] for item in data],
        "db_id": [item["db_id"] for item in data]
    })

# LƯU Ý: Load thẳng file spider-realistic.json (508 câu)
val_ds = load_spider_unified("spider_data/spider-realistic.json")

# 4. CHẠY SUY LUẬN (INFERENCE)
predictions, gold_lines = [], []

print("Bắt đầu chạy dự đoán trên tập Spider-Realistic...")
for item in val_ds:
    input_text = f"translate to SQL: {item['question']} | Schema: {get_crp_schema(item['db_id'])}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        output = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
    predictions.append(tokenizer.decode(output[0], skip_special_tokens=True) + "\n")
    gold_lines.append(f"{item['query']}\t{item['db_id']}\n")

with open('pred.txt', 'w') as f: f.writelines(predictions)
with open('gold.txt', 'w') as f: f.writelines(gold_lines)

# 5. CHẤM ĐIỂM
print("Hoàn tất dự đoán! Đang chạy script chấm điểm...")
!sed -i 's/conn = sqlite3.connect(db)/conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")/' evaluation.py
!python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all

/tmp/ipykernel_3786/3249147643.py:14: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  np.core.multiarray._reconstruct,


Mounted at /content/drive
Dataset URL: https://www.kaggle.com/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset
License(s): unknown
100% 96.0M/96.0M [00:03<00:00, 26.7MB/s]



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Đang tải mô hình từ /content/drive/My Drive/BART_Base_Spider_CRP_FFT lên cuda...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Bắt đầu chạy dự đoán trên tập Spider-Realistic...
Hoàn tất dự đoán! Đang chạy script chấm điểm...
eval_err_num:1
medium pred: SELECT avg(t1.age) ,  min(age)  ,  max(T1.Age) FROM singer AS t1 JOIN song AS t2 ON t1.song_name  =  "France"
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

medium pred: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  <=  2015
medium gold: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  =  2015

medium pred: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  <=  2015
medium gold: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  =  2015

eval_err_num:2
extra pred: SELECT T3.name ,  T2.capacity FROM stadium AS T1 JOIN concert AS T2 ON T1.stadium_id  =  t2.host_id JOIN song AS T3 ON T3:song_fuelled  >  2014 GROUP BY t1.park_id ORDER BY count(*) DESC LIMIT 1
extra gold: SELECT T2.name ,  T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id WHERE T1.year 